<a href="https://colab.research.google.com/github/MonKhach/Data_Visualization_project/blob/main/Decision_Tree_Bagging_Regression_monkhach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 01: Find the Best Regression Tree Model

In class, we used decision trees, pre-pruning, post-pruning, and bagging for classification. In this homework, transfer those ideas to regression using the California Housing dataset.

Your objective is to find the best regression-tree-based model you can. You may use any relevant method from the lesson: manual pre-pruning, grid-search pre-pruning, cost-complexity post-pruning, bagging, or a combination. Submit the completed notebook through Classroom 50.

## Homework Requirements

Complete the notebook by doing the following:

- Train one unrestricted decision tree baseline.
- Search for a better model using methods from the lesson.
- Choose one final model based on evidence, not guesswork.
- Print MAE, MSE, RMSE, and R2 for the baseline and final model.
- Explain why your final model is better than the baseline.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Load the Dataset

Load California Housing. The target is median house value in units of $100,000, so this is a regression task.

In [ ]:
housing = fetch_california_housing(as_frame=True)

X = housing.data
y = housing.target

housing_df = X.copy()
housing_df["MedHouseVal"] = y

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target: median house value in units of $100,000")
housing_df.head()

Feature matrix shape: (20640, 8)
Target shape: (20640,)
Target: median house value in units of $100,000


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [ ]:
housing_df.describe().T

,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.870671,1.899822,0.499900,2.563400,3.534800,4.743250,15.000100
HouseAge,20640.0,28.639486,12.585558,1.000000,18.000000,29.000000,37.000000,52.000000
AveRooms,20640.0,5.429000,2.474173,0.846154,4.440716,5.229129,6.052381,141.909091
AveBedrms,20640.0,1.096675,0.473911,0.333333,1.006079,1.048780,1.099526,34.066667
Population,20640.0,1425.476744,1132.462122,3.000000,787.000000,1166.000000,1725.000000,35682.000000
AveOccup,20640.0,3.070655,10.386050,0.692308,2.429741,2.818116,3.282261,1243.333333
Latitude,20640.0,35.631861,2.135952,32.540000,33.930000,34.260000,37.710000,41.950000
Longitude,20640.0,-119.569704,2.003532,-124.350000,-121.800000,-118.490000,-118.010000,-114.310000
MedHouseVal,20640.0,2.068558,1.153956,0.149990,1.196000,1.797000,2.647250,5.000010


## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Training rows: 15480
Test rows: 5160


## 4. Metric Function

Create a helper that prints regression metrics. Use it for the baseline and the final selected model.

In [ ]:
def regression_metrics(model, model_name, X_train, y_train, X_test, y_test):
    """Return train/test regression metrics for one fitted model."""
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_predictions) ## TODO: Implement

    test_mse = mean_squared_error(y_test, test_predictions) ## TODO: Implement

    return {
        "Model": model_name,
        "Train MAE": mean_absolute_error(y_train, train_predictions), ## TODO: Implement
        "Test MAE": mean_absolute_error(y_test, test_predictions), ## TODO: Implement
        "Train MSE": train_mse,
        "Test MSE": test_mse,
        "Train RMSE": np.sqrt(train_mse),
        "Test RMSE": np.sqrt(test_mse),
        "Train R2": r2_score(y_train, train_predictions), ## TODO: Implement
        "Test R2": r2_score(y_test, test_predictions), ## TODO: Implement
        "Depth": model.get_depth() if hasattr(model, "get_depth") else np.nan,
        "Leaves": model.get_n_leaves() if hasattr(model, "get_n_leaves") else np.nan,
        "OOB R2": model.oob_score_ if hasattr(model, "oob_score_") else np.nan,
    }

## 5. Baseline Model

Train an unrestricted regression tree. This is the model your final solution must improve.

In [ ]:
baseline_model = DecisionTreeRegressor(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_leaf_nodes=None
) ## TODO: Implement
baseline_model.fit(X_train, y_train)
baseline_results = regression_metrics(
    baseline_model,
    "Baseline unrestricted tree",
    X_train,
    y_train,
    X_test,
    y_test,
)

pd.DataFrame([baseline_results])

,Model,Train MAE,Test MAE,Train MSE,Test MSE,Train RMSE,Test RMSE,Train R2,Test R2,Depth,Leaves,OOB R2
0,Baseline unrestricted tree,4.131062e-17,0.469071,9.081073e-32,0.530475,3.013482e-16,0.728337,1.0,0.599102,36,14847,NaN


## 6. Find the Best Version of the Model

Use what you learned in class to find a better regression tree model. You decide the method: pruning, bagging, or a combination.

Your search should be organized enough to justify your final choice. For example, you can compare candidate models by cross-validation RMSE and then evaluate the selected model on the test set.

In [ ]:
## TODO: Implement
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

tree_param_grid = {
    'max_depth': [6, 8, 10, 12, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'ccp_alpha': [ 0.0, 0.0005, 0.001]
}

tree_search = GridSearchCV(
    estimator=DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),
    param_grid=tree_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

tree_search.fit(X_train, y_train)
best_tree = tree_search.best_estimator_
tree_cv_rmse = -tree_search.best_score_
print('Best Decision Tree parameters:')
print(tree_search.best_params_)
print('\nBest Decision Tree CV RMSE:')
print(tree_cv_rmse)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best Decision Tree parameters:
{'ccp_alpha': 0.0, 'max_depth': 12, 'min_samples_leaf': 10, 'min_samples_split': 2}

Best Decision Tree CV RMSE:
0.6307023434894818


In [ ]:
bagging_model = BaggingRegressor(
    estimator=best_tree,
    random_state=RANDOM_STATE,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1
)

bagging_param_grid = {
    'n_estimators': [50, 100],
    'max_samples': [0.7, 1.0],
    'max_features': [0.8, 1.0]
}

bagging_search = GridSearchCV(
    estimator=bagging_model,
    param_grid=bagging_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    verbose=1
)
bagging_search.fit(X_train, y_train)
best_bagging = bagging_search.best_estimator_
bagging_cv_rmse = -bagging_search.best_score_
print('Best Bagging Parameters:')
print(bagging_search.best_params_)
print('\nBest Bagging CV RMSE')
print(bagging_cv_rmse)
print('\nBagging OOB R2:')
print(best_bagging.oob_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Bagging Parameters:
{'max_features': 0.8, 'max_samples': 1.0, 'n_estimators': 100}

Best Bagging CV RMSE
0.5095914003019917

Bagging OOB R2:
0.8096478000302982


In [ ]:
model_comparision = pd.DataFrame({
    'Model': [
        'Pruned Decision Tree',
        'Bagging Regression'
    ],
    'CV RMSE': [
        tree_cv_rmse,
        bagging_cv_rmse
    ]
})

model_comparision = model_comparision.sort_values(
    by='CV RMSE'
).reset_index(drop=True)

model_comparision

,Model,CV RMSE
0,Bagging Regression,0.509591
1,Pruned Decision Tree,0.630702


## 7. Select the Final Best Model

Choose the model with the strongest evidence. Usually this means the lowest validation RMSE, then confirm its performance on the test set.

In [ ]:
if bagging_cv_rmse < tree_cv_rmse:
  final_model_name = 'Tuned Bagging Regressor'
  final_model = best_bagging
else:
  final_model_name = 'Tuned Pruned Decision Tree'
  final_model = best_tree

final_results = regression_metrics(
    final_model,
    final_model_name,
    X_train,
    y_train,
    X_test,
    y_test,
)

print("Selected final model:", final_model_name)

comparison = pd.DataFrame([baseline_results, final_results])
comparison

Selected final model: Tuned Bagging Regressor


,Model,Train MAE,Test MAE,Train MSE,Test MSE,Train RMSE,Test RMSE,Train R2,Test R2,Depth,Leaves,OOB R2
0,Baseline unrestricted tree,4.131062e-17,0.469071,9.081073e-32,0.530475,3.013482e-16,0.728337,1.000000,0.599102,36.0,14847.0,NaN
1,Tuned Bagging Regressor,2.887028e-01,0.341815,1.774151e-01,0.251304,4.212068e-01,0.501302,0.867037,0.810081,NaN,NaN,0.809648


## 8. Evaluate with Regression Metrics

Print the baseline and final model metrics. Your final model should improve test RMSE.

In [ ]:
metric_columns = [
    "Model",
    "Train MAE",
    "Test MAE",
    "Train MSE",
    "Test MSE",
    "Train RMSE",
    "Test RMSE",
    "Train R2",
    "Test R2",
    "Depth",
    "Leaves",
    "OOB R2",
]

comparison[metric_columns]

,Model,Train MAE,Test MAE,Train MSE,Test MSE,Train RMSE,Test RMSE,Train R2,Test R2,Depth,Leaves,OOB R2
0,Baseline unrestricted tree,4.131062e-17,0.469071,9.081073e-32,0.530475,3.013482e-16,0.728337,1.000000,0.599102,36.0,14847.0,NaN
1,Tuned Bagging Regressor,2.887028e-01,0.341815,1.774151e-01,0.251304,4.212068e-01,0.501302,0.867037,0.810081,NaN,NaN,0.809648
